# Regression Baselines: Linear Regression and Decision Tree Regressor
Owner: Anupama Singh

Instead of predicting yes/no, we predict **how many points a student's score grows** from fall to spring (the gain).

The paper's targets are a median split of this gain (Word ID: gain >= 27), so we can also turn the predicted gain into an AUC and compare with the classification models.

## 1. Load data and build the gain

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupKFold, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, roc_auc_score

df = pd.read_csv("../data/ECRIDataSet.csv")
raw = pd.read_csv("../data/ECRIRawData.csv", low_memory=False)

# Spring (Time 2) scores, used ONLY to build the target, never as features
spring = raw[raw["Time"] == 2.0].set_index("StuID")[["RMwidRS", "RMwdaRS"]].add_suffix("_spring")
df = df.join(spring, on="StuID")
df = df[df["RMwidRS_target"].notna()]

df["RMwidRS_gain"] = df["RMwidRS_spring"] - df["RMwidRS"]
df["RMwdaRS_gain"] = df["RMwdaRS_spring"] - df["RMwdaRS"]
df[["RMwidRS_gain", "RMwdaRS_gain"]].describe().round(1)

,RMwidRS_gain,RMwdaRS_gain
count,946.0,946.0
mean,27.4,11.9
std,8.9,7.4
min,-10.0,-12.0
25%,21.0,7.0
50%,27.0,12.0
75%,33.0,17.0
max,57.0,32.0


## 2. Features and splits (same as notebook 01)

In [2]:
features = ["Gender", "Tier", "LEP", "SpEd", "Tx", "Age1b", "Tier2_N",
            "TKPctCrct", "TYearsTeaching", "NWFcls", "NWFwrc", "ORFwc",
            "SAwrS", "SAsrS", "SAtoS", "RMwidRS", "RMwdaRS"]
X = df[features]
groups = df["SchlID"]
cv = GroupKFold(n_splits=5)

targets = {"Word Identification": "RMwidRS", "Word Attack": "RMwdaRS"}

## 3. Define the models
`Mean baseline` always predicts the average gain. Any useful model must beat it.

In [3]:
def mean_baseline():
    from sklearn.dummy import DummyRegressor
    return DummyRegressor(strategy="mean")

def linear_regression():
    return make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), LinearRegression())

def tree_regressor(depth):
    return make_pipeline(SimpleImputer(strategy="median"),
                         DecisionTreeRegressor(max_depth=depth, min_samples_leaf=20, random_state=0))

## 4. Tune Decision Tree depth (lower MAE is better)

In [4]:
for name, s in targets.items():
    y = df[s + "_gain"]
    for depth in range(2, 9):
        pred = cross_val_predict(tree_regressor(depth), X, y, cv=cv, groups=groups)
        print(f"{name:20s} depth={depth}  MAE={mean_absolute_error(y, pred):.2f}")

Word Identification  depth=2  MAE=6.93
Word Identification  depth=3  MAE=7.04
Word Identification  depth=4  MAE=7.08
Word Identification  depth=5  MAE=7.04
Word Identification  depth=6  MAE=7.02
Word Identification  depth=7  MAE=7.13
Word Identification  depth=8  MAE=7.25
Word Attack          depth=2  MAE=5.93
Word Attack          depth=3  MAE=6.02
Word Attack          depth=4  MAE=6.07
Word Attack          depth=5  MAE=6.18


Word Attack          depth=6  MAE=6.29
Word Attack          depth=7  MAE=6.35
Word Attack          depth=8  MAE=6.38


## 5. Final results
MAE / RMSE are in score points. AUC uses the predicted gain to rank students against the paper's yes/no target.

In [5]:
rows = []
for name, s in targets.items():
    y = df[s + "_gain"]
    label = df[s + "_target"]
    best_depth = min(range(2, 9), key=lambda d: mean_absolute_error(
        y, cross_val_predict(tree_regressor(d), X, y, cv=cv, groups=groups)))
    for model_name, model in [("Mean baseline", mean_baseline()),
                              ("Linear Regression", linear_regression()),
                              (f"Decision Tree Regressor (depth={best_depth})", tree_regressor(best_depth))]:
        pred = cross_val_predict(model, X, y, cv=cv, groups=groups)
        auc = roc_auc_score(label, pred) if model_name != "Mean baseline" else 0.5
        rows.append({"target": name, "model": model_name,
                     "MAE": round(mean_absolute_error(y, pred), 2),
                     "RMSE": round(mean_squared_error(y, pred) ** 0.5, 2),
                     "AUC": round(auc, 3)})

results = pd.DataFrame(rows)
results.to_csv("../results/regression_baselines.csv", index=False)
results

,target,model,MAE,RMSE,AUC
0,Word Identification,Mean baseline,7.13,8.96,0.500
1,Word Identification,Linear Regression,6.53,8.29,0.670
2,Word Identification,Decision Tree Regressor (depth=2),6.93,8.74,0.574
3,Word Attack,Mean baseline,5.94,7.39,0.500
4,Word Attack,Linear Regression,5.75,7.10,0.627
5,Word Attack,Decision Tree Regressor (depth=2),5.93,7.34,0.536


## 6. Which features matter most? (Linear Regression coefficients, Word ID)

In [6]:
model = linear_regression().fit(X, df["RMwidRS_gain"])
coefs = pd.Series(model[-1].coef_, index=features).sort_values(key=abs, ascending=False)
coefs.round(2)

RMwidRS          -3.82
RMwdaRS           1.87
SAtoS             1.73
SAsrS            -1.17
Tx                1.07
Tier              0.91
SpEd             -0.78
Tier2_N           0.76
NWFwrc            0.76
Age1b            -0.67
LEP              -0.49
Gender            0.25
TYearsTeaching    0.20
ORFwc             0.19
SAwrS            -0.15
TKPctCrct        -0.14
NWFcls            0.12
dtype: float64